# Wav2Vec2 ONNX / OpenVINO FP16 / OpenVINO INT8 Pipeline

End-to-end pipeline:
1. Export model to ONNX
2. Build calibration dataset from `zalozbadev/hsb_audio_corpus`
3. Run PTQ INT8 quantization with NNCF
4. Benchmark ORT vs OV-FP16 vs OV-INT8
5. Accuracy evaluation (CER) across all three backends

In [ ]:
from transformers import AutoModelForCTC, AutoProcessor
import librosa
import torch
import numpy as np
import time
import os

MODEL_ID    = "Korla/Wav2Vec2BertForCTC-hsb-0"
DATASET_ID  = "zalozbadev/hsb_audio_corpus"
ONNX_PATH   = "./models/wav2vec2.onnx"
OV_FP16_XML = "./models/ov_fp16/model.xml"
OV_INT8_XML = "./models/ov_int8/model.xml"
SAMPLE_RATE = 16000

# Best settings found by the tuning sweep on Intel Core Ultra 9 275HX
ORT_INTRA_THREADS = 24
OV_THREADS        = 24

print("Configuration ready.")

## 1. Load model and processor

In [ ]:
model = (
    AutoModelForCTC.from_pretrained(MODEL_ID)
    .to("cpu")
    .eval()
    .to(torch.float16)
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("Model loaded, dtype:", next(model.parameters()).dtype)

## 2. ONNX export (skipped if file already exists)

In [ ]:
from torch.export.dynamic_shapes import Dim

if os.path.exists(ONNX_PATH):
    print(f"Skipping export — {ONNX_PATH} already exists.")
else:
    dummy = torch.randn(1, SAMPLE_RATE, dtype=torch.float32)
    dummy_input = processor(dummy, sampling_rate=SAMPLE_RATE, return_tensors="pt").to(torch.float16)
    dynamic_shapes = {
        "input":  {0: 1, 1: Dim.AUTO, 2: 160},
        "output": {0: 1, 1: Dim.AUTO, 2: 41},
    }
    torch.onnx.export(
        model,
        dummy_input["input_features"],
        ONNX_PATH,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes=dynamic_shapes,
        dynamo=True,
    )
    print(f"Exported to {ONNX_PATH}.")

## 3. Build calibration and accuracy datasets from `zalozbadev/hsb_audio_corpus`

Uses streaming mode — the full 1.5 GB corpus is **not** downloaded.

In [ ]:
import datasets as hf_datasets

CALIB_SAMPLES    = 64
ACCURACY_SAMPLES = 30

dataset = hf_datasets.load_dataset(
    DATASET_ID,
    split="train",
    streaming=True,
)

calib_features   = []  # list of (1, seq_len, 160) float16 arrays
accuracy_items   = []  # list of (features_array, reference_text)

for sample in dataset.take(CALIB_SAMPLES + ACCURACY_SAMPLES):
    audio  = sample["audio"]
    text   = (sample.get("transcription") or sample.get("text") or "").strip()
    speech = np.array(audio["array"], dtype=np.float32)
    sr_in  = audio["sampling_rate"]

    if sr_in != SAMPLE_RATE:
        speech = librosa.resample(speech, orig_sr=sr_in, target_sr=SAMPLE_RATE)

    features = processor(
        speech,
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
    )["input_features"].numpy()   # (1, seq_len, 160), float16

    if len(calib_features) < CALIB_SAMPLES:
        calib_features.append(features)
    else:
        accuracy_items.append((features, text))

print(f"Calibration samples : {len(calib_features)}")
print(f"Accuracy samples    : {len(accuracy_items)}")
print(f"Example input shape : {calib_features[0].shape}, dtype: {calib_features[0].dtype}")

## 4. ONNX Runtime session (tuned)

In [ ]:
import onnxruntime as ort

so = ort.SessionOptions()
so.execution_mode            = ort.ExecutionMode.ORT_SEQUENTIAL
so.graph_optimization_level  = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
so.intra_op_num_threads      = ORT_INTRA_THREADS
so.inter_op_num_threads      = 1

ort_session = ort.InferenceSession(
    ONNX_PATH,
    sess_options=so,
    providers=["CPUExecutionProvider"],
)
print("ORT ready — input type:", ort_session.get_inputs()[0].type)

## 5. OpenVINO FP16 — build from ONNX, cache to disk

In [ ]:
import openvino as ov
import openvino.properties as props
import openvino.properties.hint as hints
import openvino.properties.intel_cpu as cpu_props

core = ov.Core()

def make_ov_config():
    return {
        hints.performance_mode:       hints.PerformanceMode.LATENCY,
        props.inference_num_threads:  OV_THREADS,
        hints.scheduling_core_type:   hints.SchedulingCoreType.ECORE_ONLY,
        hints.enable_hyper_threading: False,
        cpu_props.denormals_optimization: True,
    }

if os.path.exists(OV_FP16_XML):
    print(f"Loading cached FP16 IR from {OV_FP16_XML}.")
else:
    print(f"Building FP16 IR from ONNX and saving to {OV_FP16_XML}...")
    os.makedirs(os.path.dirname(OV_FP16_XML), exist_ok=True)
    ov.save_model(core.read_model(ONNX_PATH), OV_FP16_XML)

print("OV FP16 IR ready.")

## 6. INT8 post-training quantization with NNCF

The model stays dynamic during quantization so every calibration sample can have its own sequence length.  
`model_type=TRANSFORMER` instructs NNCF to keep sensitive attention layers in higher precision.

In [ ]:
import nncf

if os.path.exists(OV_INT8_XML):
    print(f"Loading cached INT8 IR from {OV_INT8_XML}.")
else:
    print(f"Quantizing with {len(calib_features)} calibration samples...")
    calib_dataset = nncf.Dataset(
        calib_features,
        transform_func=lambda arr: {"input": arr},
    )
    ov_int8 = nncf.quantize(
        core.read_model(ONNX_PATH),
        calib_dataset,
        preset=nncf.QuantizationPreset.PERFORMANCE,
        model_type=nncf.ModelType.TRANSFORMER,
    )
    os.makedirs(os.path.dirname(OV_INT8_XML), exist_ok=True)
    ov.save_model(ov_int8, OV_INT8_XML)
    print(f"Saved INT8 IR to {OV_INT8_XML}.")

print("OV INT8 IR ready.")

## 7. Benchmark — fixed input shape

Reshape both OV models to the exact benchmark input shape so OpenVINO can select shape-specific kernels.

In [ ]:
BENCHMARK_WAV    = "1.wav"
BENCH_DURATION_S = 10

waveform, _   = librosa.load(BENCHMARK_WAV, sr=SAMPLE_RATE)
bench_wav     = waveform[:SAMPLE_RATE * BENCH_DURATION_S]
bench_feat    = processor(bench_wav, sampling_rate=SAMPLE_RATE, return_tensors="pt").to(torch.float16)
bench_array   = bench_feat["input_features"].numpy()   # (1, seq_len, 160) float16
bench_shape   = list(bench_array.shape)
print("Benchmark input:", bench_shape, bench_array.dtype)

def _compile_fixed(xml_path):
    m = core.read_model(xml_path)
    m.reshape(bench_shape)
    return core.compile_model(m, "CPU", config=make_ov_config())

compiled_fp16 = _compile_fixed(OV_FP16_XML)
compiled_int8 = _compile_fixed(OV_INT8_XML)
print("OV models compiled for fixed shape.")

In [ ]:
WARM_UP = 5
SAMPLES = 20

def bench(fn, warm_up=WARM_UP, samples=SAMPLES):
    for _ in range(warm_up):
        fn()
    t = []
    for _ in range(samples):
        t0 = time.perf_counter()
        fn()
        t.append(time.perf_counter() - t0)
    return sum(t) / samples, min(t)

ort_avg,  ort_min  = bench(lambda: ort_session.run(None, {"input": bench_array}))
fp16_avg, fp16_min = bench(lambda: compiled_fp16([bench_array]))
int8_avg, int8_min = bench(lambda: compiled_int8([bench_array]))

print(f"{'Backend':<24} {'Avg (s)':>10} {'Min (s)':>10} {'Speedup vs ORT':>16}")
print("-" * 64)
print(f"{'ONNX Runtime (FP16)':<24} {ort_avg:>10.4f} {ort_min:>10.4f} {'1.00x':>16}")
print(f"{'OpenVINO FP16':<24} {fp16_avg:>10.4f} {fp16_min:>10.4f} {ort_avg/fp16_avg:>15.2f}x")
print(f"{'OpenVINO INT8':<24} {int8_avg:>10.4f} {int8_min:>10.4f} {ort_avg/int8_avg:>15.2f}x")

## 8. Accuracy evaluation (CER)

Dynamic-shape compilations are used here so each sample's natural sequence length is preserved.

In [ ]:
def cer(ref, hyp):
    """Character Error Rate via Levenshtein distance."""
    r, h = list(ref), list(hyp)
    dp = list(range(len(h) + 1))
    for i, rc in enumerate(r):
        prev = dp[:]
        dp[0] = i + 1
        for j, hc in enumerate(h):
            dp[j + 1] = min(prev[j + 1] + 1, dp[j] + 1, prev[j] + (rc != hc))
    return dp[len(h)] / max(len(r), 1)


def decode_logits(logit_array):
    """CTC greedy decode via processor."""
    ids = torch.from_numpy(np.array(logit_array)).argmax(dim=-1)
    if ids.ndim == 1:
        ids = ids.unsqueeze(0)
    return processor.batch_decode(ids)[0]


# Dynamic-shape models for accuracy (no reshape per sample)
fp16_dyn = core.compile_model(core.read_model(OV_FP16_XML), "CPU", config=make_ov_config())
int8_dyn = core.compile_model(core.read_model(OV_INT8_XML), "CPU", config=make_ov_config())

ort_cer_total = fp16_cer_total = int8_cer_total = 0.0
rows = []

for features, ref_text in accuracy_items:
    if not ref_text:
        continue

    ort_logits  = ort_session.run(None, {"input": features})[0][0]   # (seq_len, vocab)
    fp16_logits = fp16_dyn([features])[0][0]
    int8_logits = int8_dyn([features])[0][0]

    ort_text  = decode_logits(ort_logits[np.newaxis])
    fp16_text = decode_logits(fp16_logits[np.newaxis])
    int8_text = decode_logits(int8_logits[np.newaxis])

    ref_l = ref_text.lower()
    ort_cer_total  += cer(ref_l, ort_text.lower())
    fp16_cer_total += cer(ref_l, fp16_text.lower())
    int8_cer_total += cer(ref_l, int8_text.lower())

    rows.append((ref_text, ort_text, fp16_text, int8_text))

n = len(rows)
print(f"Evaluated {n} samples.\n")
print(f"{'Backend':<24} {'Mean CER':>10}")
print("-" * 36)
print(f"{'ONNX Runtime (FP16)':<24} {ort_cer_total/n:>10.3f}")
print(f"{'OpenVINO FP16':<24} {fp16_cer_total/n:>10.3f}")
print(f"{'OpenVINO INT8':<24} {int8_cer_total/n:>10.3f}")

In [ ]:
# Side-by-side transcription examples
header = f"{'REF':<40} {'ORT':<40} {'OV-FP16':<40} {'OV-INT8':<40}"
print(header)
print("-" * len(header))
for ref, ort_t, fp16_t, int8_t in rows[:10]:
    print(f"{ref[:38]:<40} {ort_t[:38]:<40} {fp16_t[:38]:<40} {int8_t[:38]:<40}")

## 9. (Optional) Thread / core-type sweep

Re-discovers the best ORT and OpenVINO CPU settings on the current machine.  
The best results found on **Intel Core Ultra 9 275HX** are already encoded as defaults above.

In [ ]:
import onnxruntime

thread_candidates = sorted({1, 4, 8, 12, os.cpu_count() or 1})
core_candidates = [
    ("ANY_CORE",   hints.SchedulingCoreType.ANY_CORE),
    ("PCORE_ONLY", hints.SchedulingCoreType.PCORE_ONLY),
    ("ECORE_ONLY", hints.SchedulingCoreType.ECORE_ONLY),
]
ht_candidates = [True, False]

def _time_fn(fn, runs=5, warm_up=3):
    for _ in range(warm_up):
        fn()
    t = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        t.append(time.perf_counter() - t0)
    return round(sum(t) / runs, 4)

# --- ORT sweep ---
ort_sweep = []
for thr in thread_candidates:
    so = onnxruntime.SessionOptions()
    so.execution_mode           = onnxruntime.ExecutionMode.ORT_SEQUENTIAL
    so.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL
    so.intra_op_num_threads     = thr
    so.inter_op_num_threads     = 1
    s = onnxruntime.InferenceSession(ONNX_PATH, sess_options=so, providers=["CPUExecutionProvider"])
    ort_sweep.append({"threads": thr, "avg_s": _time_fn(lambda ss=s: ss.run(None, {"input": bench_array}))})

# --- OV sweep ---
ov_sweep = []
for thr in thread_candidates:
    for ht in ht_candidates:
        for cname, ctype in core_candidates:
            try:
                m = core.read_model(ONNX_PATH)
                m.reshape(bench_shape)
                c = core.compile_model(m, "CPU", config={
                    hints.performance_mode:       hints.PerformanceMode.LATENCY,
                    props.inference_num_threads:  thr,
                    hints.scheduling_core_type:   ctype,
                    hints.enable_hyper_threading: ht,
                    cpu_props.denormals_optimization: True,
                })
                ov_sweep.append({"threads": thr, "ht": ht, "core": cname,
                                  "avg_s": _time_fn(lambda cc=c: cc([bench_array]))})
            except RuntimeError as e:
                ov_sweep.append({"threads": thr, "ht": ht, "core": cname, "avg_s": None, "error": str(e)})

print("Best ORT:", min(ort_sweep, key=lambda x: x["avg_s"]))
print("Best OV: ", min((r for r in ov_sweep if r["avg_s"]), key=lambda x: x["avg_s"]))
print("\nAll ORT:"); [print(r) for r in sorted(ort_sweep, key=lambda x: x["avg_s"])]
print("\nTop-10 OV:"); [print(r) for r in sorted((r for r in ov_sweep if r["avg_s"]), key=lambda x: x["avg_s"])[:10]]